# Perform illumination correction and resave ND2 images as TIFF

This recipe is an enhanced version of the plain re-save notebook with additional correction of uneven illumination and dark field images.

Specifically, we perform correction (for each YX plane in the dataset) as follows:

$I_{corrected} = (I_{raw} - I_{dark}) / I_{flat}$

Here, $I_{dark}$ is an image recorded without any illumination and $I_{flat}$ is a *normalized* flatfield image made by averaging several images of a uniformly dye-coated sample and then divided by the mean (resulting in values centering around 1, so low values will be raised via the correction, whereas high values will be lowered).

## Input

This recipe expects a base input folder (plus optional raw data subfolder) containing multidimensional ```.nd2``` files.

Additionally, you have to specify a path to a json file containing the correction information and correction images (a bright and dark field image for each wavelength and objective). The json file should have the following structure:

```json
[
    {
        "objective": "Plan Apo λ 100x Oil",
        "darkfields": {
            "405": "relative_path_to_darkfield_image_405.tif",
            "488": "relative_path_to_darkfield_image_488.tif",
            ... (other wavelengths)
        },
        "flatfields": {
            "405": "relative_path_to_flatfield_image_405.tif",
            ... (other wavelengths)
        }
    },
    ... (other objectives)
]
```

Optionally, you can specify along which dimensions to split input files (by default, it will be channels ```"C"```, time points ```"T"``` and positions ```"P"```) and a name for the output folder.

## Output

For each position and channel of each ```.nd2``` image in the input folder, a ```.tif``` file will be created in a output folder (which is placed in the base input directory). The name of output files will be ```{input file name (without ending)}_ch{channel number}_pos{position number}.tif``` (only dimensions that are present in the file will show up in the output filename).

## Helper functions

In [ ]:
from pathlib import Path
from tifffile import imread
import json
import nd2
import numpy as np
import xarray
import re
import functools

from calmutils.imageio import save_tiff_imagej
from nd2_resaving import xarray_to_tiffs


def get_objective_name_from_nd2(fd):
    # if already opened, use as-is (don't close)
    if isinstance(fd, nd2.ND2File):
        meta = fd.metadata
    # otherwise open on-the-fly
    else:
        with nd2.ND2File(fd) as reader:
            meta = reader.metadata
    # get objective Name for first channel
    # NOTE: we assume the same objective is used throughout experiment
    return meta.channels[0].microscope.objectiveName


def get_wavelengths(reader):

    # if reader is not an open ND2File, try to open and call recursively
    if not isinstance(reader, nd2.ND2File):
        with nd2.ND2File(reader) as reader_i:
            return get_wavelengths(reader_i)

    # extraction of wavelength from text_info:
    pattern = r"(?:^|\n) +Line:\d;.*?(?=\n *Line:\d;|\n *(?=\S)|$)"
    matches = re.findall(pattern, reader.text_info['description'], re.DOTALL | re.MULTILINE)
    wavelengths = []
    for match in matches:
        if 'On' in match:
            wavelength = re.search(r'ExW:(\d+);', match).group(1)
            wavelengths.append(wavelength)
    return wavelengths


def load_correction_images(correction_info_path, obj, wls):

    with open(correction_info_path) as fd:
        correction_info = json.load(fd)

    for correction_info_i in correction_info:

        # skip non-matching objective
        if correction_info_i['objective'] != obj:
            continue

        darkfields = []
        flatfields = []

        for wl in wls:
            # error on non-existing images
            if wl not in correction_info_i['darkfields'] or wl not in correction_info_i['flatfields']:
                raise ValueError(f'No correction image for wavelength {wl} for objective {obj} found.')

            darkfield_i = imread(Path(correction_info_path).parent / correction_info_i['darkfields'][wl])

            # load flatfield, normalize to mean=1
            flatfield_i = imread(Path(correction_info_path).parent / correction_info_i['flatfields'][wl])
            flatfield_i /= flatfield_i.mean()

            darkfields.append(darkfield_i)
            flatfields.append(flatfield_i)

        # stack along C, to xarray
        darkfields = np.stack(darkfields)
        flatfields = np.stack(flatfields)
        darkfields = xarray.DataArray(darkfields, dims=['C', 'Y', 'X'])
        flatfields = xarray.DataArray(flatfields, dims=['C', 'Y', 'X'])
        # TODO: squeeze to handle single-channel?

        return flatfields, darkfields

    # error on no matching objective
    raise ValueError(f'No illumination correction for objective {obj} found.')


# cache load
# (we are likely to always have the same objective, wavelengths in one experiment so only cache one call)
@functools.lru_cache(1)
def load_correction_images_cached(correction_info_path, obj, wls):
    return load_correction_images(correction_info_path, obj, wls)

# Parameters

In [ ]:
# base path + subdirectory of raw data
in_path = '/Users/josefin/PhD/weihua_UHRF1_CellCycle/images/2603011/'
in_subdirectory = 'raw'

# where to save (relative to in_path)
out_subdirectory = 'tif_illumination_corrected'

# path to correction info JSON (and correction images)
correction_info_path = '/Users/josefin/PhD/weihua_UHRF1_CellCycle/models/FlatfieldCorrection/illumination_correction_spinning_disk.json'

In [ ]:
# glob input, show for verification
in_files = sorted((Path(in_path) / in_subdirectory).glob('*.nd2'))
in_files

In [ ]:
# make outdir if it does not exist
out_dir = Path(in_path) / out_subdirectory
if not out_dir.exists():
    out_dir.mkdir()

for in_file in in_files:

    # get flatfields and darkfields
    objective = get_objective_name_from_nd2(in_file)
    wavelengths = tuple(get_wavelengths(in_file))
    flatfields, darkfields = load_correction_images_cached(correction_info_path, objective, wavelengths)

    img = nd2.imread(in_file, xarray=True)

    # correct and clip to original dtype
    img_corrected = (img - darkfields) / flatfields
    img_corrected = np.clip(img_corrected, np.iinfo(img.dtype).min, np.iinfo(img.dtype).max).astype(img.dtype)

    xarray_to_tiffs(img_corrected, out_dir, in_file.stem)

    print(f'processed {in_file}.')

## Testing code below

In [ ]:

with open(correction_info_path) as fd:
        correction_info = json.load(fd)

correction_info

In [ ]:
import matplotlib.pyplot as plt

for correction_info_i in correction_info:

    fig, axs = plt.subplots(ncols=4, figsize=(12, 4))
    fig.suptitle(correction_info_i['objective'])

    for ax, (wavelength, correction_image_path) in zip(axs, correction_info_i['flatfields'].items()):
        
        img = imread(Path(correction_info_path).parent / correction_image_path)
        ax.imshow(img, cmap='gray')
        ax.set_title(wavelength)

In [ ]:

get_objective_name_from_nd2('/Users/david/Desktop/20240903-c2c12-DAPI-40x001.nd2')


In [ ]:

get_wavelengths('/Users/david/Desktop/20240823-mESC-DAPI-001.nd2')

In [ ]:
img = nd2.imread('/Users/david/Desktop/20240823-mESC-DAPI-001.nd2', xarray=True)

xarray_to_tiffs(img, '/Users/david/Desktop/test1', 'img')